In [ ]:
%pip install openai pydantic python-dotenv

In [2]:
import os
from typing import List
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

MODEL="llama-3.1-8b-instant"

In [3]:

# define schema using pydantic
class Person(BaseModel):
    name: str = Field(description="Full name of the individual.")
    date_of_birth: str = Field(description="Date of birth text, e.g., November 7, 1867.")
    occupation: str = Field(description="Primary job, research field, or title.")
    nationality: str = Field(description="Country of origin or historical citizenship.")

class PeopleList(BaseModel):
    # wrapper model to make LLM output an array of structured entities
    people: List[Person]

def main():
    # unstructured given text for extraction
    unstructured_biography = """
    Marie Curie, born on November 7, 1867 in Warsaw, Poland, was a pioneering physicist 
    and chemist who conducted groundbreaking research on radioactivity. She was the first 
    woman to win a Nobel Prize. Her colleague Albert Einstein, a German-born theoretical 
    physicist born on March 14, 1879, developed the theory of relativity. Meanwhile, 
    Ada Lovelace, an English mathematician born on December 10, 1815, is often regarded 
    as the first computer programmer for her work on Charles Babbage's Analytical Engine.
    """

    print("--- Unstructured Biography Text ---")
    print(unstructured_biography.strip())
    print("\n--- Requesting Structured Token Extraction from Groq ---")

    # make instructions to work with our pydantic model.
    #give exact schema for the json output which can be parsed by pydantic
    system_instruction = (
        "You are a strict data extraction engine. Extract all historical people mentioned "
        "in the text with their explicit details. You MUST respond exclusively with a valid "
        "JSON object matching this exact schema layout:\n"
        "{\n"
        "  'people': [\n"
        "    { 'name': string, 'date_of_birth': string, 'occupation': string, 'nationality': string }\n"
        "  ]\n"
        "}"
    )

    # get a response with json turned on
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": unstructured_biography}
        ],
        response_format={"type": "json_object"},
        temperature=0.0  # no variations
    )

    raw_output_string = response.choices[0].message.content

    try:
        # this parses the json and gives a clean python object structure
        parsed_data = PeopleList.model_validate_json(raw_output_string)
        
        print("\n✅ Extraction Successful! Verified Type-Safe Output:\n")
        
        # iterate over the parsed pydantic objects
        for person in parsed_data.people:
            print(f"👤 Name:        {person.name}")
            print(f"   Born:        {person.date_of_birth}")
            print(f"   Occupation:  {person.occupation}")
            print(f"   Nationality: {person.nationality}")
            print("-" * 50)

    except Exception as parsing_error:
        print(f"\n❌ Pipeline Parsing Failure: {parsing_error}")
        print(f"Raw response text: {raw_output_string}")

if __name__ == "__main__":
    main()

--- Unstructured Biography Text ---
Marie Curie, born on November 7, 1867 in Warsaw, Poland, was a pioneering physicist 
    and chemist who conducted groundbreaking research on radioactivity. She was the first 
    woman to win a Nobel Prize. Her colleague Albert Einstein, a German-born theoretical 
    physicist born on March 14, 1879, developed the theory of relativity. Meanwhile, 
    Ada Lovelace, an English mathematician born on December 10, 1815, is often regarded 
    as the first computer programmer for her work on Charles Babbage's Analytical Engine.

--- Requesting Structured Token Extraction from Groq ---

✅ Extraction Successful! Verified Type-Safe Output:

👤 Name:        Marie Curie
   Born:        November 7, 1867
   Occupation:  physicist and chemist
   Nationality: Polish
--------------------------------------------------
👤 Name:        Albert Einstein
   Born:        March 14, 1879
   Occupation:  theoretical physicist
   Nationality: German
--------------------------